## Computational Cost Analysis — PI-PINN vs. Baseline PINN (Real Dengue Data)
### Heliyon Reviewer Requirement (Section 5)
> *"A practical advantage of PI-PINN is its simplicity. Measure: training time,
> number of epochs to convergence, GPU memory consumption. Demonstrating negligible
> computational overhead would strengthen the practical contribution."*

**This notebook is fully self-contained** — it does not depend on any other notebook.
Just run all cells top-to-bottom. Only requirement: `dengue_srilanka_2017.csv` must be
in your Google Drive at `MyDrive/dengue_thesis/dengue_srilanka_2017.csv`.

It trains two groups, each with the same 20 seeds, same architecture, same data:
- **PI-PINN** — full loss (data + physics + Gaussian MAP prior, λ3=0.1)
- **Baseline PINN** — no prior (λ3=0), otherwise identical

**Early stopping has been removed.** Both groups now train for the exact same
**fixed budget of 3000 epochs** — so "epochs to converge" is no longer a metric
in this notebook. It instead records, per run: **wall-clock training time**,
**mean per-epoch time**, and **peak GPU memory**. It then reports Mean±SD for both
groups plus Welch's t-test, Mann-Whitney U test, and Bootstrap 95% CI for each cost
metric, a **parameter recovery stability analysis** (β/γ coefficient of variation,
% of runs within a literature-plausible band, and Levene's test for equality of
variances), and a paper-ready comparison table.

> **Important interpretive note:** because both groups now share the exact same
> fixed 3000-epoch budget, total training time is no longer confounded by early
> stopping — it is simply 3000 × the per-epoch cost of each group. However, a
> fixed epoch budget by itself still cannot confirm that β/γ have converged to a
> physically plausible estimate (real-data PINN inverse problems are only weakly
> identifiable). So this notebook separates the **per-epoch cost** (the confound-free
> measure of the prior term's raw computational overhead) from **which group's
> β/γ actually converge better**, which is answered directly by the parameter
> recovery stability analysis (Cell 12 / Table 2) — that is the cell to check for
> "which model performs better on beta/gamma convergence."

| Cell | Content |
|------|---------|
| 1 | Imports |
| 2 | Google Drive mount + Data load (chronological split) |
| 3 | Device + hyperparameters & config (literature prior, λ weights, fixed epoch budget) |
| 4 | SIRPINN model class |
| 5 | Helper functions (tensor, dataloader) + shared tensors |
| 6 | Training function (fixed-epoch, no early stopping, per-epoch timing) + held-out RMSE helper |
| 7 | Multi-run helper functions |
| 8 | **PI-PINN loop — 20 seeds** (records time, per-epoch time, GPU memory) |
| 9 | **Baseline loop (λ3=0) — 20 seeds** (records time, per-epoch time, GPU memory) |
| 10 | Cost statistics (Mean±SD) + Welch's t-test / Mann-Whitney U, incl. per-epoch time |
| 11 | Bootstrap 95% CI (per-group + PI-PINN − Baseline difference), incl. per-epoch time |
| 12 | **Parameter Recovery Stability**: β/γ CV, literature-plausible-band %, Levene's test — ← answers "which model converges better" |
| 13 | Paper-ready computational cost table (Table 1) + stability table (Table 2) |
| 14 | Results & discussion paragraph (auto-filled) |


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import time
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.init import xavier_uniform_
from scipy import stats
from scipy.stats import bootstrap

print("\u2713 All libraries imported.")
print(f"  PyTorch version : {torch.__version__}")
print(f"  CUDA available  : {torch.cuda.is_available()}")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Load & prepare REAL dengue data (chronological split) ──────────
def load_and_prepare_data(test_frac=0.1, val_frac=0.1):
    """
    Loads real Sri Lanka dengue (2017) weekly S/I/R fractions.
    Uses a CHRONOLOGICAL 80/10/10 split (NOT random train_test_split):
    this is real time-series surveillance data, and a random split would
    leak future information into training (temporal leakage).
    """
    data_loaded = pd.read_csv('/content/drive/MyDrive/dengue_thesis/dengue_srilanka_2017.csv')
    data_loaded = data_loaded.sort_values('time').reset_index(drop=True)

    t_all = data_loaded['time'].values.reshape(-1, 1)
    S_all = data_loaded['Susceptible'].values.reshape(-1, 1)
    I_all = data_loaded['Infected'].values.reshape(-1, 1)
    R_all = data_loaded['Recovered'].values.reshape(-1, 1)

    n       = len(data_loaded)
    n_test  = max(1, int(round(n * test_frac)))
    n_val   = max(1, int(round(n * val_frac)))
    n_train = n - n_test - n_val

    train_sl = slice(0, n_train)
    val_sl   = slice(n_train, n_train + n_val)
    test_sl  = slice(n_train + n_val, n)

    return (t_all[train_sl], t_all[val_sl], t_all[test_sl],
            S_all[train_sl], I_all[train_sl], R_all[train_sl],
            S_all[val_sl],   I_all[val_sl],   R_all[val_sl],
            S_all[test_sl],  I_all[test_sl],  R_all[test_sl])

(t_train_set, t_val_set, t_test_set,
 data_S_train_set, data_I_train_set, data_R_train_set,
 data_S_val_set,   data_I_val_set,   data_R_val_set,
 data_S_test_set,  data_I_test_set,  data_R_test_set) = load_and_prepare_data()

print(f"\u2713 Data loaded (real dengue, Sri Lanka 2017).")
print(f"  Train : {t_train_set.shape[0]} samples")
print(f"  Val   : {t_val_set.shape[0]} samples")
print(f"  Test  : {t_test_set.shape[0]} samples")


In [ ]:
# ── Device ───────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"  Device: {device}")

# ── Literature-informed prior (PI-PINN only) — NOT ground truth ───
MU_BETA     = 0.145   # prior mean for beta
SIGMA_BETA  = 0.05    # prior std  for beta
MU_GAMMA    = 0.101   # prior mean for gamma
SIGMA_GAMMA = 0.03    # prior std  for gamma

# ── Loss weights (PI-PINN); baseline below reuses lambda1/lambda2 with lambda3=0
LAMBDA1 = 0.6   # data loss weight
LAMBDA2 = 0.3   # physics loss weight
LAMBDA3 = 0.1   # prior loss weight

# ── Training hyperparameters ──────────────────────────────────────
BATCH_SIZE  = 32
EPOCHS_MR   = 3000   # fixed epoch budget — early stopping removed, every run trains this many epochs

# ── Multi-run config ──────────────────────────────────────────────
N_RUNS = 20
SEEDS  = [42, 123, 7, 2024, 999, 17, 55, 88, 314, 271,
          101, 202, 303, 404, 505, 606, 707, 808, 909, 1001]

print(f"\n  Prior  : mu_beta={MU_BETA}, sigma_beta={SIGMA_BETA}  |  "
      f"mu_gamma={MU_GAMMA}, sigma_gamma={SIGMA_GAMMA}")
print(f"  N_RUNS = {N_RUNS}  |  Seeds  = {SEEDS}")
print(f"  Epochs = {EPOCHS_MR} (fixed, no early stopping)")


In [ ]:
class SIRPINN(nn.Module):
    """
    Physics-Informed Neural Network for SIR parameter estimation.
    Architecture: Input(1) -> [Linear->Tanh] x 3 -> Output(3)
    Learnable SIR parameters: beta, gamma (via nn.ParameterDict)
    """
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim,      hidden_dims[0]), nn.Tanh(),
            nn.Linear(hidden_dims[0], hidden_dims[1]), nn.Tanh(),
            nn.Linear(hidden_dims[1], hidden_dims[2]), nn.Tanh(),
            nn.Linear(hidden_dims[2], output_dim)
        )

        for layer in self.network:
            if isinstance(layer, nn.Linear):
                xavier_uniform_(layer.weight)
                nn.init.constant_(layer.bias, 0)

        self.params = nn.ParameterDict({
            'beta' : nn.Parameter(torch.tensor(1.0, dtype=torch.float32)),
            'gamma': nn.Parameter(torch.tensor(1.0, dtype=torch.float32))
        })

    def forward(self, t):
        return self.network(t)

print("\u2713 SIRPINN class defined.")
print("  Architecture : 1 -> 50 -> 50 -> 50 -> 3  (Tanh activations)")
print("  Learnable    : beta (init=1.0), gamma (init=1.0)")


In [ ]:
def to_tensor(x, device, requires_grad=False):
    """Convert numpy array to float32 tensor on device."""
    return torch.tensor(x, dtype=torch.float32,
                        requires_grad=requires_grad).to(device)

def prepare_dataloader(t, S, I, R, batch_size, device):
    """Build a shuffled DataLoader from numpy arrays."""
    t_t = to_tensor(t, device, requires_grad=True)
    S_t = to_tensor(S, device)
    I_t = to_tensor(I, device)
    R_t = to_tensor(R, device)
    ds  = TensorDataset(t_t, S_t, I_t, R_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=True)

# ── Prepare shared train/val/test tensors (used across all runs) ──
t_train_tensor      = to_tensor(t_train_set,      device, requires_grad=True)
data_S_train_tensor = to_tensor(data_S_train_set, device)
data_I_train_tensor = to_tensor(data_I_train_set, device)
data_R_train_tensor = to_tensor(data_R_train_set, device)

t_val_tensor        = to_tensor(t_val_set,        device)
data_S_val_tensor   = to_tensor(data_S_val_set,   device)
data_I_val_tensor   = to_tensor(data_I_val_set,   device)
data_R_val_tensor   = to_tensor(data_R_val_set,   device)

t_test_tensor       = to_tensor(t_test_set,       device)
data_S_test_tensor  = to_tensor(data_S_test_set,  device)
data_I_test_tensor  = to_tensor(data_I_test_set,  device)
data_R_test_tensor  = to_tensor(data_R_test_set,  device)

print("\u2713 Helper functions defined.")
print("  Shared train/val/test tensors ready.")


In [ ]:
def train_with_physics_loss_sir_v2(
        model, optimizer, train_loader_local,
        epochs,
        t_train_tensor, data_S_train_tensor, data_I_train_tensor, data_R_train_tensor,
        t_val_tensor,   data_S_val_tensor,   data_I_val_tensor,   data_R_val_tensor,
        mu_beta_prior,  sigma_beta_prior,
        mu_gamma_prior, sigma_gamma_prior,
        lambda1=0.6, lambda2=0.3, lambda3=0.1,
        verbose=False):
    """
    PI-PINN training with MAP-based Gaussian prior loss.
    Total loss = lambda1*L_data + lambda2*L_physics + lambda3*L_prior
    Prior loss = (beta-mu_beta)^2/(2*sigma_beta^2)
               + (gamma-mu_gamma)^2/(2*sigma_gamma^2)
    Setting lambda3=0 disables the prior term entirely (the "Baseline PINN"
    configuration used for comparison in this notebook).

    NOTE: early stopping has been removed. The model always trains for the
    full fixed `epochs` budget, and the returned parameter estimates are
    simply whatever beta/gamma the model has reached after the final epoch
    (no best-validation-loss checkpoint restoring).
    """
    train_losses        = []
    val_losses          = []
    parameter_estimates = {key: [] for key in model.params.keys()}
    epoch_times          = []   # wall-clock seconds for each individual epoch

    for epoch in range(epochs):
        if device.type == 'cuda':
            torch.cuda.synchronize()
        _epoch_t0 = time.time()

        model.train()
        batch_losses = []

        for t_batch, S_batch, I_batch, R_batch in train_loader_local:
            optimizer.zero_grad()
            t_batch = t_batch.requires_grad_(True)

            out    = model(t_batch)
            S_pred = out[:, 0:1]
            I_pred = out[:, 1:2]
            R_pred = out[:, 2:3]

            beta  = model.params['beta']
            gamma = model.params['gamma']

            dS_dt = torch.autograd.grad(S_pred, t_batch,
                        grad_outputs=torch.ones_like(S_pred), create_graph=True)[0]
            dI_dt = torch.autograd.grad(I_pred, t_batch,
                        grad_outputs=torch.ones_like(I_pred), create_graph=True)[0]
            dR_dt = torch.autograd.grad(R_pred, t_batch,
                        grad_outputs=torch.ones_like(R_pred), create_graph=True)[0]

            physics_loss = (torch.mean((dS_dt + beta * S_pred * I_pred)**2) +
                            torch.mean((dI_dt - beta * S_pred * I_pred + gamma * I_pred)**2) +
                            torch.mean((dR_dt - gamma * I_pred)**2))

            data_loss = (torch.mean((S_pred - S_batch)**2) +
                         torch.mean((I_pred - I_batch)**2) +
                         torch.mean((R_pred - R_batch)**2))

            prior_loss = ((beta  - mu_beta_prior )**2 / (2 * sigma_beta_prior**2)  +
                          (gamma - mu_gamma_prior)**2 / (2 * sigma_gamma_prior**2))

            loss = lambda2 * physics_loss + lambda1 * data_loss + lambda3 * prior_loss
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())

        epoch_loss = sum(batch_losses) / len(batch_losses)
        train_losses.append(epoch_loss)

        for key in parameter_estimates:
            parameter_estimates[key].append(model.params[key].item())

        model.eval()
        with torch.no_grad():
            out_val  = model(t_val_tensor)
            val_loss = (torch.mean((out_val[:,0:1] - data_S_val_tensor)**2) +
                        torch.mean((out_val[:,1:2] - data_I_val_tensor)**2) +
                        torch.mean((out_val[:,2:3] - data_R_val_tensor)**2))
            val_losses.append(val_loss.item())

        if device.type == 'cuda':
            torch.cuda.synchronize()
        epoch_times.append(time.time() - _epoch_t0)

        if verbose and epoch % 500 == 0:
            print(f'  Epoch {epoch:5d} | beta={beta.item():.5f} | '
                  f'gamma={gamma.item():.6f} | val={val_loss.item():.6f}')

    return train_losses, val_losses, parameter_estimates, epoch_times


def test_rmse(model, t_test_tensor, S_test_t, I_test_t, R_test_t):
    """Held-out RMSE on real, unseen dengue data."""
    model.eval()
    with torch.no_grad():
        out = model(t_test_tensor)
        rmse = torch.sqrt(torch.mean((out[:,0:1] - S_test_t)**2) +
                           torch.mean((out[:,1:2] - I_test_t)**2) +
                           torch.mean((out[:,2:3] - R_test_t)**2))
    return rmse.item()

print("\u2713 train_with_physics_loss_sir_v2() and test_rmse() defined (early stopping removed, fixed-epoch training).")


In [ ]:
def get_fresh_model_optimizer(seed: int):
    """Fresh SIRPINN + Adam, fully seeded for reproducibility."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    m   = SIRPINN(input_dim=1, hidden_dims=[50, 50, 50], output_dim=3).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    return m, opt

def get_fresh_loader(seed: int):
    """DataLoader with seed-controlled shuffling."""
    torch.manual_seed(seed)
    return prepare_dataloader(
        t_train_set, data_S_train_set, data_I_train_set, data_R_train_set,
        BATCH_SIZE, device)

print("\u2713 Multi-run helper functions defined.")
print(f"  Ready to run {N_RUNS} independent experiments per group (PI-PINN + Baseline).")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# PI-PINN — 20 Independent Runs — Real Dengue Data
# Records wall-clock training time and peak GPU memory per run
# ══════════════════════════════════════════════════════════════════

beta_finals  = []
gamma_finals = []
rmse_finals  = []
epochs_ran   = []
train_times  = []   # wall-clock seconds per run
peak_mem_mb  = []   # peak GPU memory (MB) per run (0 if no GPU)
per_epoch_time = []  # mean wall-clock seconds PER EPOCH per run (warm-up epoch excluded)

print("=" * 65)
print(f"  Starting {N_RUNS} PI-PINN runs (with prior) — real dengue data")
print(f"  Epochs={EPOCHS_MR} (fixed, no early stopping)  Batch={BATCH_SIZE}")
print("=" * 65)

for run_idx, seed in enumerate(SEEDS):
    print(f"\n[PI-PINN Run {run_idx + 1:02d}/{N_RUNS}]  seed = {seed}")

    m_r, opt_r = get_fresh_model_optimizer(seed)
    loader_r   = get_fresh_loader(seed)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device)
    t_start = time.time()

    _, _, p_est, ep_times = train_with_physics_loss_sir_v2(
        model               = m_r,
        optimizer           = opt_r,
        train_loader_local  = loader_r,
        epochs              = EPOCHS_MR,
        t_train_tensor      = t_train_tensor,
        data_S_train_tensor = data_S_train_tensor,
        data_I_train_tensor = data_I_train_tensor,
        data_R_train_tensor = data_R_train_tensor,
        t_val_tensor        = t_val_tensor,
        data_S_val_tensor   = data_S_val_tensor,
        data_I_val_tensor   = data_I_val_tensor,
        data_R_val_tensor   = data_R_val_tensor,
        mu_beta_prior       = MU_BETA,
        sigma_beta_prior    = SIGMA_BETA,
        mu_gamma_prior      = MU_GAMMA,
        sigma_gamma_prior   = SIGMA_GAMMA,
        lambda1=LAMBDA1, lambda2=LAMBDA2, lambda3=LAMBDA3,
        verbose=True
    )

    elapsed = time.time() - t_start
    train_times.append(elapsed)
    peak_mem_mb.append(torch.cuda.max_memory_allocated(device) / 1024**2
                        if torch.cuda.is_available() else 0.0)
    # per-epoch cost, excluding epoch 0 (one-time CUDA kernel/cache warm-up)
    per_epoch_time.append(np.mean(ep_times[1:]) if len(ep_times) > 1 else ep_times[0])

    fb = p_est['beta'][-1]
    fg = p_est['gamma'][-1]
    beta_finals.append(fb)
    gamma_finals.append(fg)
    epochs_ran.append(len(p_est['beta']))
    rmse_finals.append(test_rmse(m_r, t_test_tensor,
                                  data_S_test_tensor, data_I_test_tensor, data_R_test_tensor))

    print(f"  \u2192 beta_hat = {fb:.5f}  |  gamma_hat = {fg:.6f}  |  "
          f"test RMSE = {rmse_finals[-1]:.6f}  |  epochs = {epochs_ran[-1]}  |  "
          f"time = {elapsed:.1f}s  |  time/epoch = {per_epoch_time[-1]*1000:.2f}ms  |  "
          f"peak GPU mem = {peak_mem_mb[-1]:.1f} MB")

print("\n\u2713 All PI-PINN runs complete.")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# BASELINE PINN (no prior, λ3 = 0) — 20 Independent Runs
# Same architecture, same real dengue data, same seeds, same λ1/λ2 as
# PI-PINN — only the Gaussian MAP prior term is switched off.
# Records wall-clock training time and peak GPU memory per run.
# ══════════════════════════════════════════════════════════════════

beta_finals_bl  = []
gamma_finals_bl = []
rmse_finals_bl  = []
epochs_ran_bl   = []
train_times_bl  = []   # wall-clock seconds per run
peak_mem_mb_bl  = []   # peak GPU memory (MB) per run (0 if no GPU)
per_epoch_time_bl = []  # mean wall-clock seconds PER EPOCH per run (warm-up epoch excluded)

print("=" * 65)
print(f"  Starting {N_RUNS} baseline runs (no prior, lambda3=0) — real dengue data")
print(f"  Epochs={EPOCHS_MR} (fixed, no early stopping)  Batch={BATCH_SIZE}")
print("=" * 65)

for run_idx, seed in enumerate(SEEDS):
    print(f"\n[Baseline Run {run_idx + 1:02d}/{N_RUNS}]  seed = {seed}")

    m_b, opt_b = get_fresh_model_optimizer(seed)
    loader_b   = get_fresh_loader(seed)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device)
    t_start = time.time()

    _, _, p_est_b, ep_times_b = train_with_physics_loss_sir_v2(
        model               = m_b,
        optimizer           = opt_b,
        train_loader_local  = loader_b,
        epochs              = EPOCHS_MR,
        t_train_tensor      = t_train_tensor,
        data_S_train_tensor = data_S_train_tensor,
        data_I_train_tensor = data_I_train_tensor,
        data_R_train_tensor = data_R_train_tensor,
        t_val_tensor        = t_val_tensor,
        data_S_val_tensor   = data_S_val_tensor,
        data_I_val_tensor   = data_I_val_tensor,
        data_R_val_tensor   = data_R_val_tensor,
        mu_beta_prior       = MU_BETA,      # unused when lambda3=0, kept for signature
        sigma_beta_prior    = SIGMA_BETA,
        mu_gamma_prior      = MU_GAMMA,
        sigma_gamma_prior   = SIGMA_GAMMA,
        lambda1=LAMBDA1, lambda2=LAMBDA2, lambda3=0.0,   # <-- prior OFF (baseline)
        verbose=True
    )

    elapsed = time.time() - t_start
    train_times_bl.append(elapsed)
    peak_mem_mb_bl.append(torch.cuda.max_memory_allocated(device) / 1024**2
                           if torch.cuda.is_available() else 0.0)
    # per-epoch cost, excluding epoch 0 (one-time CUDA kernel/cache warm-up)
    per_epoch_time_bl.append(np.mean(ep_times_b[1:]) if len(ep_times_b) > 1 else ep_times_b[0])

    fb = p_est_b['beta'][-1]
    fg = p_est_b['gamma'][-1]
    beta_finals_bl.append(fb)
    gamma_finals_bl.append(fg)
    epochs_ran_bl.append(len(p_est_b['beta']))
    rmse_finals_bl.append(test_rmse(m_b, t_test_tensor,
                                     data_S_test_tensor, data_I_test_tensor, data_R_test_tensor))

    print(f"  \u2192 beta_hat = {fb:.5f}  |  gamma_hat = {fg:.6f}  |  "
          f"test RMSE = {rmse_finals_bl[-1]:.6f}  |  epochs = {epochs_ran_bl[-1]}  |  "
          f"time = {elapsed:.1f}s  |  time/epoch = {per_epoch_time_bl[-1]*1000:.2f}ms  |  "
          f"peak GPU mem = {peak_mem_mb_bl[-1]:.1f} MB")

print("\n\u2713 All baseline runs complete.")


In [ ]:
# ── Computational Cost Analysis: PI-PINN vs. Baseline (no prior) ─
time_arr       = np.array(train_times)
time_arr_bl    = np.array(train_times_bl)
mem_arr        = np.array(peak_mem_mb)
mem_arr_bl     = np.array(peak_mem_mb_bl)
per_epoch_arr    = np.array(per_epoch_time)
per_epoch_arr_bl = np.array(per_epoch_time_bl)

def summarize(arr, name):
    mn, sd = np.mean(arr), np.std(arr, ddof=1)
    print(f"  {name:<28}: {mn:.5f} +/- {sd:.5f}")
    return mn, sd

print("PI-PINN (with prior):")
time_mean,      time_std      = summarize(time_arr,      "Training time (s)")
mem_mean,       mem_std       = summarize(mem_arr,       "Peak GPU memory (MB)")
per_epoch_mean, per_epoch_std = summarize(per_epoch_arr, "Time per epoch (s)")

print("\nBaseline (no prior):")
time_mean_bl,      time_std_bl      = summarize(time_arr_bl,      "Training time (s)")
mem_mean_bl,       mem_std_bl       = summarize(mem_arr_bl,       "Peak GPU memory (MB)")
per_epoch_mean_bl, per_epoch_std_bl = summarize(per_epoch_arr_bl, "Time per epoch (s)")

def compare_groups(a, b, name):
    """Welch's t-test (unequal variance) + Mann-Whitney U (non-parametric)."""
    t_stat, t_p = stats.ttest_ind(a, b, equal_var=False)             # Welch's t-test
    u_stat, u_p = stats.mannwhitneyu(a, b, alternative='two-sided')  # Mann-Whitney U
    pooled_sd = np.sqrt((np.var(a, ddof=1) + np.var(b, ddof=1)) / 2)
    cohens_d  = (np.mean(a) - np.mean(b)) / pooled_sd if pooled_sd > 0 else float('nan')
    return {'metric': name, 'welch_t': t_stat, 'welch_p': t_p,
            'mwu_u': u_stat, 'mwu_p': u_p, 'cohens_d': cohens_d}

results_time      = compare_groups(time_arr,      time_arr_bl,      'Training time')
results_mem       = compare_groups(mem_arr,       mem_arr_bl,       'Peak GPU memory')
results_per_epoch = compare_groups(per_epoch_arr, per_epoch_arr_bl, 'Time per epoch')

for r in (results_time, results_mem, results_per_epoch):
    sig = "significant (p<0.05)" if r['welch_p'] < 0.05 else "not significant (p>=0.05)"
    print(f"\n  {r['metric']}:")
    print(f"    Welch's t-test      : t={r['welch_t']:.3f}, p={r['welch_p']:.4g}  -> {sig}")
    print(f"    Mann-Whitney U test : U={r['mwu_u']:.1f}, p={r['mwu_p']:.4g}")
    print(f"    Cohen's d           : {r['cohens_d']:.3f}")

time_overhead_pct      = (time_mean - time_mean_bl) / time_mean_bl * 100
per_epoch_overhead_pct = (per_epoch_mean - per_epoch_mean_bl) / per_epoch_mean_bl * 100
print(f"\n\u2713 Computational cost statistics computed.")
print(f"  PI-PINN TOTAL training-time overhead vs. baseline : {time_overhead_pct:+.2f}%")
print(f"  PI-PINN PER-EPOCH (per-iteration) overhead        : {per_epoch_overhead_pct:+.2f}%")
print(f"\n  NOTE: early stopping has been removed — every run (both groups) trains for")
print(f"  the exact same fixed budget of {EPOCHS_MR} epochs, so total training time and")
print(f"  per-epoch time now measure the SAME underlying quantity (no early-stopping")
print(f"  confound). Whether PI-PINN's prior term actually helps beta/gamma converge to")
print(f"  a better answer within those {EPOCHS_MR} epochs is a separate question, answered")
print(f"  by the Parameter Recovery Stability analysis below (Cell 12 / Table 2).")


In [ ]:
# ── Bootstrap 95% CI (percentile method, 10,000 resamples) ────────
RNG_SEED_BOOT = 0

def bootstrap_mean_ci(sample, n_resamples=10000, seed=RNG_SEED_BOOT):
    res = bootstrap((sample,), np.mean, n_resamples=n_resamples,
                     confidence_level=0.95, method='percentile',
                     random_state=seed)
    return res.confidence_interval.low, res.confidence_interval.high

def bootstrap_diff_ci(sample_a, sample_b, n_resamples=10000, seed=RNG_SEED_BOOT):
    """Bootstrap CI for mean(sample_a) - mean(sample_b), unpaired resampling."""
    def diff_of_means(x, y, axis=-1):
        return np.mean(x, axis=axis) - np.mean(y, axis=axis)
    res = bootstrap((sample_a, sample_b), diff_of_means, n_resamples=n_resamples,
                     confidence_level=0.95, method='percentile',
                     paired=False, random_state=seed)
    return res.confidence_interval.low, res.confidence_interval.high

time_boot_ci_pinn   = bootstrap_mean_ci(time_arr)
time_boot_ci_bl      = bootstrap_mean_ci(time_arr_bl)
mem_boot_ci_pinn     = bootstrap_mean_ci(mem_arr)
mem_boot_ci_bl       = bootstrap_mean_ci(mem_arr_bl)
per_epoch_boot_ci_pinn = bootstrap_mean_ci(per_epoch_arr)
per_epoch_boot_ci_bl   = bootstrap_mean_ci(per_epoch_arr_bl)

time_boot_ci_diff      = bootstrap_diff_ci(time_arr,      time_arr_bl)
mem_boot_ci_diff       = bootstrap_diff_ci(mem_arr,       mem_arr_bl)
per_epoch_boot_ci_diff = bootstrap_diff_ci(per_epoch_arr, per_epoch_arr_bl)

print("\u2713 Bootstrap 95% CIs computed (10,000 resamples, percentile method).")
print(f"\n  Training time (s)   PI-PINN CI : ({time_boot_ci_pinn[0]:.2f}, {time_boot_ci_pinn[1]:.2f})")
print(f"  Training time (s)   Baseline CI: ({time_boot_ci_bl[0]:.2f}, {time_boot_ci_bl[1]:.2f})")
print(f"  Training time (s)   Diff CI    : ({time_boot_ci_diff[0]:.2f}, {time_boot_ci_diff[1]:.2f})")
print(f"\n  Peak GPU mem (MB)   PI-PINN CI : ({mem_boot_ci_pinn[0]:.1f}, {mem_boot_ci_pinn[1]:.1f})")
print(f"  Peak GPU mem (MB)   Baseline CI: ({mem_boot_ci_bl[0]:.1f}, {mem_boot_ci_bl[1]:.1f})")
print(f"  Peak GPU mem (MB)   Diff CI    : ({mem_boot_ci_diff[0]:.1f}, {mem_boot_ci_diff[1]:.1f})")
print(f"\n  Time / epoch (s)    PI-PINN CI : ({per_epoch_boot_ci_pinn[0]:.5f}, {per_epoch_boot_ci_pinn[1]:.5f})")
print(f"  Time / epoch (s)    Baseline CI: ({per_epoch_boot_ci_bl[0]:.5f}, {per_epoch_boot_ci_bl[1]:.5f})")
print(f"  Time / epoch (s)    Diff CI    : ({per_epoch_boot_ci_diff[0]:.5f}, {per_epoch_boot_ci_diff[1]:.5f})")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Parameter Recovery Stability & Literature-Plausible-Range Analysis
# ══════════════════════════════════════════════════════════════════
# WHY THIS CELL EXISTS: both groups now train for the exact same FIXED
# epoch budget (early stopping removed), so "epochs to converge" is no
# longer a meaningful comparison between them. What actually matters is
# whether the final beta/gamma each group lands on after those fixed
# epochs is close to the epidemiologically plausible literature range,
# and reproducible across seeds. This is a classic identifiability gap
# in PINN inverse problems: many (beta, gamma) combinations can fit
# S/I/R trajectories almost equally well, especially on a single real
# outbreak's noisy weekly counts -- so THIS cell (not training time) is
# the actual answer to "which model, PI-PINN or Baseline, converges
# better on beta/gamma."
#
# To check this directly we report, per group, using the final beta/gamma
# estimate from every one of the N_RUNS seeds:
#   - mean +/- SD of the estimate
#   - coefficient of variation (CV = SD/mean): a scale-free run-to-run
#     stability score -- higher CV means the estimate is less reproducible
#     across random seeds/initializations
#   - the % of runs whose estimate falls inside a literature-plausible band
#     (prior mean +/- 2*prior SD, i.e. ~95% mass under the literature-informed
#     Gaussian prior already used for PI-PINN's mu_beta/sigma_beta etc.)
#   - Levene's test for equality of variances between PI-PINN and Baseline
#     (non-parametric-friendly alternative to an F-test for comparing spread)

beta_arr     = np.array(beta_finals)
beta_arr_bl  = np.array(beta_finals_bl)
gamma_arr    = np.array(gamma_finals)
gamma_arr_bl = np.array(gamma_finals_bl)

BETA_LO,  BETA_HI  = MU_BETA  - 2 * SIGMA_BETA,  MU_BETA  + 2 * SIGMA_BETA
GAMMA_LO, GAMMA_HI = MU_GAMMA - 2 * SIGMA_GAMMA, MU_GAMMA + 2 * SIGMA_GAMMA

def cv(arr):
    """Coefficient of variation (SD/mean); scale-free stability measure."""
    m = np.mean(arr)
    return np.std(arr, ddof=1) / m if m != 0 else float('nan')

def pct_in_range(arr, lo, hi):
    return 100.0 * np.mean((arr >= lo) & (arr <= hi))

def param_stability_summary(name, arr, arr_bl, lo, hi):
    m,  s  = np.mean(arr),    np.std(arr,    ddof=1)
    mb, sb = np.mean(arr_bl), np.std(arr_bl, ddof=1)
    in_rng,  in_rng_bl = pct_in_range(arr, lo, hi), pct_in_range(arr_bl, lo, hi)
    lev_stat, lev_p = stats.levene(arr, arr_bl)

    print(f"\n  {name}  (literature-plausible band: [{lo:.4f}, {hi:.4f}]):")
    print(f"    PI-PINN  : {m:.5f} +/- {s:.5f}   CV={cv(arr):.3f}   in-band: {in_rng:.1f}%")
    print(f"    Baseline : {mb:.5f} +/- {sb:.5f}   CV={cv(arr_bl):.3f}   in-band: {in_rng_bl:.1f}%")
    sig = "variances differ significantly (p<0.05)" if lev_p < 0.05 else "no significant variance difference (p>=0.05)"
    print(f"    Levene's test (equal variance) : W={lev_stat:.3f}, p={lev_p:.4g}  -> {sig}")

    return {'name': name, 'mean': m, 'std': s, 'cv': cv(arr), 'pct_in_range': in_rng,
             'mean_bl': mb, 'std_bl': sb, 'cv_bl': cv(arr_bl), 'pct_in_range_bl': in_rng_bl,
             'levene_stat': lev_stat, 'levene_p': lev_p}

print("=" * 92)
print("  PARAMETER RECOVERY STABILITY  (N =", N_RUNS, "runs per group, real dengue data)")
print("=" * 92)

beta_stability  = param_stability_summary("beta",  beta_arr,  beta_arr_bl,  BETA_LO,  BETA_HI)
gamma_stability = param_stability_summary("gamma", gamma_arr, gamma_arr_bl, GAMMA_LO, GAMMA_HI)

print("\n" + "=" * 92)
print("  Interpretation: a higher CV and/or lower in-band %% for a group means that")
print("  group's beta/gamma estimates are less reproducible across seeds and/or less")
print("  epidemiologically plausible -- exactly the identifiability gap the")
print("  literature-informed prior is designed to close. Whichever group shows the")
print("  LOWER CV and HIGHER in-band %% for beta and gamma (ideally backed by a")
print("  significant Levene's test) is the one with the BETTER beta/gamma convergence")
print("  within the same fixed epoch budget.")

better_beta  = "PI-PINN" if beta_stability['cv']  < beta_stability['cv_bl']  else "Baseline"
better_gamma = "PI-PINN" if gamma_stability['cv'] < gamma_stability['cv_bl'] else "Baseline"
print(f"\n  \u2192 Better beta  convergence (lower CV): {better_beta}")
print(f"  \u2192 Better gamma convergence (lower CV): {better_gamma}")


In [ ]:
print("\n" + "=" * 92)
print("  TABLE 1: Computational Cost \u2014 PI-PINN (with prior) vs. Baseline PINN (lambda3=0)")
print(f"  (N = {N_RUNS} runs per group, real dengue data, Sri Lanka 2017, fixed {EPOCHS_MR}-epoch budget)")
print("=" * 92)
print(f"  {'Metric':<24}{'PI-PINN (mean+/-SD)':>22}{'Baseline (mean+/-SD)':>22}"
      f"{'Welch p':>12}{'MWU p':>10}")
print("-" * 92)
print(f"  {'Training time (s)':<24}{f'{time_mean:.2f}+/-{time_std:.2f}':>22}"
      f"{f'{time_mean_bl:.2f}+/-{time_std_bl:.2f}':>22}"
      f"{results_time['welch_p']:>12.4g}{results_time['mwu_p']:>10.4g}")
print(f"  {'Peak GPU memory (MB)':<24}{f'{mem_mean:.1f}+/-{mem_std:.1f}':>22}"
      f"{f'{mem_mean_bl:.1f}+/-{mem_std_bl:.1f}':>22}"
      f"{results_mem['welch_p']:>12.4g}{results_mem['mwu_p']:>10.4g}")
print(f"  {'Time / epoch (s)':<24}{f'{per_epoch_mean:.5f}+/-{per_epoch_std:.5f}':>22}"
      f"{f'{per_epoch_mean_bl:.5f}+/-{per_epoch_std_bl:.5f}':>22}"
      f"{results_per_epoch['welch_p']:>12.4g}{results_per_epoch['mwu_p']:>10.4g}")
print("=" * 92)
print(f"  Epochs are fixed at {EPOCHS_MR} for every run (no early stopping) \u2014 see TABLE 2")
print("  below for how well each group's beta/gamma actually converged.")
print(f"\n  PI-PINN TOTAL training-time overhead vs. baseline : {time_overhead_pct:+.2f}%")
print(f"  PI-PINN PER-EPOCH overhead vs. baseline            : {per_epoch_overhead_pct:+.2f}%")
print(f"  Bootstrap 95% CI, total-time diff (s)              : "
      f"({time_boot_ci_diff[0]:.2f}, {time_boot_ci_diff[1]:.2f})")
print(f"  Bootstrap 95% CI, peak-memory diff (MB)             : "
      f"({mem_boot_ci_diff[0]:.1f}, {mem_boot_ci_diff[1]:.1f})")
print(f"  Bootstrap 95% CI, per-epoch time diff (s)           : "
      f"({per_epoch_boot_ci_diff[0]:.5f}, {per_epoch_boot_ci_diff[1]:.5f})")

print("\n" + "=" * 92)
print("  TABLE 2: Parameter Recovery Stability (final beta/gamma across N runs)")
print("=" * 92)
print(f"  {'Parameter':<12}{'Group':<12}{'Mean+/-SD':>20}{'CV':>8}{'In-band %':>12}{'Levene p':>12}")
print("-" * 92)
print(f"  {'beta':<12}{'PI-PINN':<12}{f'{beta_stability["mean"]:.5f}+/-{beta_stability["std"]:.5f}':>20}"
      f"{beta_stability['cv']:>8.3f}{beta_stability['pct_in_range']:>11.1f}%{beta_stability['levene_p']:>12.4g}")
print(f"  {'':<12}{'Baseline':<12}{f'{beta_stability["mean_bl"]:.5f}+/-{beta_stability["std_bl"]:.5f}':>20}"
      f"{beta_stability['cv_bl']:>8.3f}{beta_stability['pct_in_range_bl']:>11.1f}%{'':>12}")
print(f"  {'gamma':<12}{'PI-PINN':<12}{f'{gamma_stability["mean"]:.5f}+/-{gamma_stability["std"]:.5f}':>20}"
      f"{gamma_stability['cv']:>8.3f}{gamma_stability['pct_in_range']:>11.1f}%{gamma_stability['levene_p']:>12.4g}")
print(f"  {'':<12}{'Baseline':<12}{f'{gamma_stability["mean_bl"]:.5f}+/-{gamma_stability["std_bl"]:.5f}':>20}"
      f"{gamma_stability['cv_bl']:>8.3f}{gamma_stability['pct_in_range_bl']:>11.1f}%{'':>12}")
print("=" * 92)
print("  In-band % = share of runs whose final estimate falls within the")
print("  literature-plausible band (prior mean +/- 2*prior SD).")

print("\n  Interpretation: the prior loss term adds a single extra scalar computation")
print("  per batch on top of the existing data+physics loss, so PER-EPOCH overhead")
print("  should be near-negligible \u2014 check TABLE 1's per-epoch row and its bootstrap")
print('  CI (a CI straddling 0 supports "negligible per-step overhead"). TABLE 2 is')
print("  where the real convergence-quality comparison lives: whichever group shows")
print("  LOWER CV and HIGHER in-band %% for beta and gamma (ideally with a significant")
print(f"  Levene's test) converges better within the same fixed {EPOCHS_MR}-epoch budget.")
print(f"\n  \u2192 Better beta  convergence: {better_beta}")
print(f"  \u2192 Better gamma convergence: {better_gamma}")


In [ ]:
print(f"""
{chr(0x2501)*70}
  RESULTS & DISCUSSION PARAGRAPH \u2014 Computational Cost (copy into your paper)
{chr(0x2501)*70}

To assess the practical overhead introduced by the Gaussian MAP prior term, we
compared PI-PINN against an otherwise identical baseline PINN (lambda3 = 0, prior
switched off) across {N_RUNS} independent seeds on real dengue surveillance data,
measuring wall-clock training time, mean per-epoch time, and peak GPU memory
consumption. Both groups were trained for the same FIXED budget of {EPOCHS_MR}
epochs (early stopping removed), so total training time is simply {EPOCHS_MR}
times the per-epoch cost of each group -- there is no early-stopping confound.

At the per-epoch level -- the measure directly attributable to the prior term's
extra computation -- PI-PINN required {per_epoch_mean:.5f} +/- {per_epoch_std:.5f} s/epoch
versus {per_epoch_mean_bl:.5f} +/- {per_epoch_std_bl:.5f} s/epoch for the baseline
(Welch's t={results_per_epoch['welch_t']:.3f}, p={results_per_epoch['welch_p']:.4g}),
an overhead of {per_epoch_overhead_pct:+.2f}%. Peak GPU memory consumption was
{mem_mean:.1f} +/- {mem_std:.1f} MB for PI-PINN versus {mem_mean_bl:.1f} +/- {mem_std_bl:.1f} MB
for the baseline (Welch's p={results_mem['welch_p']:.4g}). At the aggregate level,
PI-PINN required {time_mean:.2f} +/- {time_std:.2f} s to train for the full
{EPOCHS_MR} epochs, versus {time_mean_bl:.2f} +/- {time_std_bl:.2f} s for the baseline
(Welch's p={results_time['welch_p']:.4g}).

Because both groups train for an identical fixed epoch budget, raw training time
alone says nothing about which group actually converged to a physically correct
(beta, gamma) -- that question is instead answered directly by parameter recovery
stability across the same {N_RUNS} seeds.

We therefore additionally report parameter recovery stability across the same
{N_RUNS} seeds. PI-PINN's beta estimates had a coefficient of variation of
{beta_stability['cv']:.3f} versus {beta_stability['cv_bl']:.3f} for the baseline, with
{beta_stability['pct_in_range']:.1f}% vs. {beta_stability['pct_in_range_bl']:.1f}% of runs
falling inside the literature-plausible band; gamma showed a CV of
{gamma_stability['cv']:.3f} vs. {gamma_stability['cv_bl']:.3f}, with
{gamma_stability['pct_in_range']:.1f}% vs. {gamma_stability['pct_in_range_bl']:.1f}% of runs
in-band (Levene's p={beta_stability['levene_p']:.4g} for beta,
p={gamma_stability['levene_p']:.4g} for gamma). On these criteria, {better_beta} shows the
better beta convergence and {better_gamma} shows the better gamma convergence.

[NOTE: fill in the sentence below only after running the notebook end-to-end and
confirming the actual numbers support it -- do not assume the direction in advance.]

Together these results indicate that the Gaussian prior term adds only a
{"negligible" if abs(per_epoch_overhead_pct) < 5 else "modest"} per-epoch computational
cost, while {"substantially improving" if beta_stability['cv'] < beta_stability['cv_bl'] else "not clearly improving"}
the run-to-run stability and physical plausibility of the recovered epidemiological
parameters relative to the baseline within the same fixed {EPOCHS_MR}-epoch budget --
supporting PI-PINN's practical simplicity as a lightweight, MAP-based alternative to
fully Bayesian approaches (e.g. B-PINNs using HMC/VI) for real-data epidemic parameter
estimation.
{chr(0x2501)*33}
""")
